In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# 프리미티브 소개

<details>
<summary><b>패키지 버전</b></summary>

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
이 버전 이상을 사용하시기를 권장합니다.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
> **Note:** 새로운 실행 모델의 베타 릴리스가 이제 제공됩니다. 지향형 실행 모델은 오류 완화 워크플로를 커스터마이징할 때 더 많은 유연성을 제공합니다. 자세한 내용은 [지향형 실행 모델](/guides/directed-execution-model) 가이드를 참조하세요.

<span id="qpu-access-patterns"></span>
## Qiskit이 프리미티브를 도입한 이유는 무엇인가요?
고전 컴퓨터 초창기에 개발자들이 CPU 레지스터를 직접 조작해야 했던 것과 유사하게, QPU에 대한 초기 인터페이스는 단순히 제어 전자 장치에서 원시 데이터를 반환했습니다.
QPU가 연구실에만 있었고 연구자들만 직접 접근할 수 있었을 때는 큰 문제가 없었습니다.
대부분의 개발자들이 이러한 원시 데이터를 0과 1로 변환하는 방법을 알지 못해도 되고 알 필요도 없다는 점을 인식하여, Qiskit은 클라우드에서 QPU에 접근하기 위한 첫 번째 추상화인 `backend.run`을 도입했습니다. 이를 통해 개발자들은 익숙한 데이터 형식으로 작업하고 더 큰 그림에 집중할 수 있게 되었습니다.

QPU에 대한 접근이 더욱 광범위해지고 더 많은 양자 알고리즘이 개발됨에 따라, 다시 한번 더 높은 수준의 추상화에 대한 필요성이 생겨났습니다. 이에 대응하여 Qiskit은 양자 알고리즘 개발의 두 가지 핵심 작업인 기댓값 추정(`Estimator`)과 Circuit 샘플링(`Sampler`)에 최적화된 프리미티브 인터페이스를 도입했습니다. 목표는 다시 한번 개발자들이 데이터 변환보다 혁신에 더 집중할 수 있도록 돕는 것입니다. 프리미티브 인터페이스는 `backend.run` 인터페이스를 대체하는데, `Sampler`가 `backend.run`이 제공했던 것과 동일한 직접 하드웨어 접근을 제공하기 때문입니다.

## 프리미티브란 무엇인가요?
컴퓨팅 시스템은 여러 추상화 계층 위에 구축됩니다. 추상화를 통해 당면한 작업과 관련된 특정 수준의 세부 사항에 집중할 수 있습니다. 하드웨어에 가까울수록 더 낮은 수준의 추상화가 필요합니다(예를 들어, CPU 명령 수준에서 데이터를 이동하거나 조작해야 할 수도 있습니다). 수행하려는 작업이 복잡할수록 추상화 수준은 더 높아집니다(예를 들어, 대수 계산을 수행하기 위해 프로그래밍 라이브러리를 사용할 수 있습니다).

이러한 맥락에서 *프리미티브*란 가장 작은 처리 명령, 즉 특정 추상화 수준에서 유용한 무언가를 만들 수 있는 가장 단순한 구성 요소입니다.

양자 컴퓨팅의 최근 발전으로 인해 더 높은 수준의 추상화에서 작업해야 할 필요성이 증가했습니다.
이 분야가 더 큰 양자 처리 장치(QPU)와 더 복잡한 워크플로를 향해 나아가면서, 개별 Qubit 신호와의 상호작용에서 벗어나 양자 장치를 필요한 작업을 수행하는 시스템으로 바라보는 방향으로 초점이 이동하고 있습니다.

양자 컴퓨터의 가장 일반적인 두 가지 작업은 양자 상태 샘플링과 기댓값 계산입니다.
이러한 작업들이 Qiskit 프리미티브인 **Estimator**와 **Sampler**의 설계를 이끌었습니다.

- Estimator는 양자 Circuit으로 준비된 상태에 대한 관측 가능량의 기댓값을 계산합니다.
- Sampler는 양자 Circuit 실행의 출력 레지스터를 샘플링합니다.

요컨대, Qiskit 프리미티브가 도입한 계산 모델은 양자 프로그래밍을 오늘날의 고전 프로그래밍에 한 걸음 더 가깝게 이동시키며, 하드웨어 세부 사항보다 달성하려는 결과에 더 집중할 수 있게 합니다.

## 프리미티브 정의 및 구현
Qiskit 프리미티브에는 두 가지 유형이 있습니다: 기본 클래스와 그 구현입니다. Qiskit 프리미티브는 Qiskit SDK의 [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives) 모듈에 있는 오픈 소스 프리미티브 기본 클래스로 정의됩니다. 제공업체(Qiskit Runtime 등)는 이러한 기본 클래스를 사용하여 자체 Sampler 및 Estimator 구현을 파생할 수 있습니다. 대부분의 사용자는 기본 프리미티브가 아닌 제공업체 구현과 상호작용하게 됩니다.

### 기본 클래스
[`BaseEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV2)와 [`BaseSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseSamplerV2) - 프리미티브 구현을 위한 공통 인터페이스를 정의하는 추상 기본 클래스입니다. [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives) 모듈의 다른 모든 클래스는 이 기본 클래스에서 상속됩니다. 특정 제공업체를 위한 프리미티브 기반 실행 모델을 직접 만들고자 하는 개발자들은 이 클래스를 사용해야 합니다. 이 클래스들은 고도로 커스터마이즈된 처리를 원하지만 기존 프리미티브 구현이 너무 단순하다고 느끼는 분들에게도 유용할 수 있습니다. 일반 사용자는 기본 클래스를 직접 사용하지 않습니다.

<span id="implementations"></span>
### 구현
다음은 프리미티브 기본 클래스의 구현입니다:

- Qiskit Runtime 프리미티브([`EstimatorV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2)와 [`SamplerV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2))는 클라우드 기반 서비스로 더욱 정교한 구현(예: 오류 완화 포함)을 제공합니다. 이 기본 프리미티브의 구현은 IBM Quantum&reg; 하드웨어에 접근하는 데 사용됩니다. IBM Qiskit Runtime을 통해 접근할 수 있습니다.

- [`StatevectorEstimator`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.StatevectorEstimator)와 [`StatevectorSampler`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.StatevectorSampler#statevectorsampler) - Qiskit에 내장된 시뮬레이터를 사용하는 프리미티브의 참조 구현입니다. Qiskit [`quantum_info`](https://docs.quantum.ibm.com/api/qiskit/quantum_info#quantum-information) 모듈로 구축되어 이상적인 상태벡터 시뮬레이션 기반 결과를 생성합니다. Qiskit을 통해 접근할 수 있습니다.

- [`BackendEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendEstimatorV2)와 [`BackendSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendSamplerV2) - 이 클래스들을 사용하여 모든 양자 컴퓨팅 리소스를 프리미티브로 "래핑"할 수 있습니다. 이를 통해 아직 프리미티브 기반 인터페이스가 없는 제공업체에 대해 프리미티브 스타일의 코드를 작성할 수 있습니다. 이 클래스들은 일반 Sampler 및 Estimator와 동일하게 사용할 수 있으며, 실행할 양자 컴퓨터를 선택하기 위한 추가적인 `backend` 인수로 초기화해야 합니다. Qiskit을 사용하여 접근할 수 있습니다.

## Qiskit 프리미티브의 장점
프리미티브를 사용하면 Qiskit 사용자가 모든 세부 사항을 명시적으로 관리하지 않고도 특정 QPU에 대한 양자 코드를 작성할 수 있습니다. 또한 추가적인 추상화 계층으로 인해 특정 제공업체의 고급 하드웨어 기능에 더 쉽게 접근할 수 있습니다. 예를 들어, Qiskit Runtime 프리미티브를 사용하면 이러한 기술들을 직접 구현하는 대신 프리미티브의 [`resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level) 같은 옵션을 토글하여 최신 오류 완화 및 억제 기술의 발전을 활용할 수 있습니다.

하드웨어 제공업체의 경우, 프리미티브를 기본적으로 구현하면 고급 후처리 기술과 같은 하드웨어 기능에 접근하는 "즉시 사용 가능한" 방법을 사용자에게 제공할 수 있습니다. 따라서 사용자가 하드웨어의 최상의 기능을 활용하기가 더 쉬워집니다.

## 프리미티브 세부 사항
앞서 설명한 것처럼 모든 프리미티브는 기본 클래스에서 생성되므로 동일한 일반적인 구조와 사용법을 가집니다. 예를 들어, 모든 Estimator 프리미티브의 입력 형식은 동일합니다. 그러나 각 구현을 고유하게 만드는 차이점이 있습니다.

> **Note:** 대부분의 사용자가 Qiskit Runtime 프리미티브에 접근하므로, 이 섹션의 나머지 예제들은 Qiskit Runtime 프리미티브를 기반으로 합니다.

<span id="estimator"></span>
### Estimator
Estimator 프리미티브는 양자 Circuit으로 준비된 상태에 대한 하나 이상의 관측 가능량의 기댓값을 계산합니다. Circuit은 매개변수화될 수 있으며, 이 경우 매개변수 값도 프리미티브에 입력으로 제공되어야 합니다.

입력은 [PUB](/guides/primitive-input-output#pubs)의 배열입니다. 각 PUB의 형식은 다음과 같습니다:

(`<단일 Circuit>`, `<하나 이상의 관측 가능량>`, `<선택적 하나 이상의 매개변수 값>`, `<선택적 정밀도>`),

여기서 선택적 `parameter values`는 목록이거나 단일 매개변수일 수 있습니다. 다양한 Estimator 구현은 다양한 구성 옵션을 지원합니다. 입력에 측정값이 포함된 경우 무시됩니다.

출력은 쌍별로 계산된 기댓값과 표준 오차를 `PubResult` 형태로 포함하는 [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult#pubresult)입니다. 각 `PubResult`에는 데이터와 메타데이터가 모두 포함됩니다.

Estimator는 [프리미티브 입력 및 출력](primitive-input-output#broadcasting-rules) 주제에 설명된 NumPy 브로드캐스팅 규칙을 따라 관측 가능량과 매개변수 값의 요소들을 결합합니다.

예제:

In [1]:
# This cell is hidden from users, it creates the circuits and observables to run

from qiskit_ibm_runtime import EstimatorV2, SamplerV2, QiskitRuntimeService
from qiskit.circuit.random import random_circuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
import numpy as np

service = QiskitRuntimeService()
backend = service.least_busy()
phi = Parameter("phi")

circuit1 = random_circuit(10, 5, seed=12345)
circuit1.rzz(phi, 1, 2)
observable1 = SparsePauliOp.from_sparse_list(
    [("ZXYZ", [1, 2, 3, 4], 1)], num_qubits=10
)
param_values1 = np.random.uniform(size=5).T

circuit2 = random_circuit(10, 5, seed=12345)
circuit2.rzz(phi, 1, 2)
observable2 = SparsePauliOp.from_sparse_list(
    [("XZYX", [1, 2, 3, 4], 1)], num_qubits=10
)
param_values2 = np.random.uniform(size=5).T

shots1 = 164
shots2 = 1024

pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
circuit1 = pm.run(circuit1)
circuit2 = pm.run(circuit2)
observable1 = observable1.apply_layout(circuit1.layout)
observable2 = observable2.apply_layout(circuit2.layout)

In [2]:
estimator = EstimatorV2(mode=backend)
estimator_job = estimator.run(
    [
        (circuit1, observable1, param_values1),
        (circuit2, observable2, param_values2),
    ]
)

<span id="sampler"></span>
### Sampler
Sampler의 핵심 작업은 하나 이상의 양자 Circuit 실행에서 출력 레지스터를 샘플링하는 것입니다. 입력 Circuit은 매개변수화될 수 있으며, 이 경우 매개변수 값도 프리미티브에 입력으로 제공되어야 합니다.

입력은 하나 이상의 [PUB](/guides/primitive-input-output#pubs)이며, 형식은 다음과 같습니다:

(`<단일 Circuit>`, `<선택적 하나 이상의 매개변수 값>`, `<선택적 shots 수>`),

여기서 여러 `parameter values` 항목이 있을 수 있으며, 선택한 Circuit에 따라 각 항목은 배열이거나 단일 매개변수일 수 있습니다. 또한 입력에는 반드시 측정값이 포함되어야 합니다.

출력은 가중치 없는 [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult#pubresult) 객체 형태의 카운트 또는 shot별 측정값입니다. 그러나 결과 클래스에는 카운트와 같이 가중치가 적용된 샘플을 반환하는 메서드가 있습니다. 전체 세부 정보는 [프리미티브 입력 및 출력](primitive-input-output#broadcasting-rules)을 참조하세요.

예제:

In [3]:
# This cell is hidden from users, add measurement instructions to circuits
circuit1.measure_active()
circuit2.measure_active()

In [4]:
sampler = SamplerV2(mode=backend)
sampler_job = sampler.run(
    [
        (circuit1, param_values1, shots1),
        (circuit2, param_values2, shots2),
    ]
)

## 다음 단계
> **Tip:** - [프리미티브 시작하기](get-started-with-primitives)를 읽고 작업에 프리미티브를 구현하세요.
>     - 자세한 [프리미티브 예제](primitives-examples)를 검토하세요.
>     - IBM Quantum Learning의 [비용 함수 레슨](/learning/courses/variational-algorithm-design/cost-functions)을 통해 프리미티브를 연습하세요.
>     - [EstimatorV2 API 참조](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2)와 [SamplerV2 API 참조](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2)를 확인하세요.
>     - [V2 프리미티브로 마이그레이션](/guides/v2-primitives)을 읽어보세요.